# Module 3.1: The Encoder Layer (The LEGO Brick)

We've built all the raw materials: Tensors, Embeddings, Positional Encoding, and Multi-Head Attention. Now, it's time to build the core architecture block. 

A full Transformer (like BERT) is just 12 or 24 of these **Encoder Blocks** stacked directly on top of each other. But we can't just stack Attention on top of Attention; the math would explode. We need stabilizers.

## 1. The Missing Pieces

### The Concept
Attention is incredibly powerful for *finding context*, but it has two major flaws:
1. It is mathematically unstable. Repeated matrix multiplications will cause numbers to vanish to zero or explode to infinity.
2. It doesn't actually "think." Attention just moves vectors around so they communicate. It doesn't process that newly combined information.

### Why do we need stabilizers?
To make deep neural networks trainable, we need to guarantee that the gradients during Backpropagation flow smoothly back to the first layer, and we need dedicated layers for memorizing facts.

## 2. Residual Connections (The Highway)

### The Analogy (The Editor)
Imagine you write a draft of an essay. You give it to an editor (the Attention Network). The editor completely rewrites the essay from scratch. Sometimes they make it better, but sometimes they ruin everything you wrote! 

Instead, what if the editor just hands you a list of *edits* (the residuals), and you simply **add** those edits to your original draft? If the edits are bad, you can easily ignore them and keep the original.

### The Math
$$ Output = Input + \text{Sublayer}(Input) $$
In code, this is literally just `out = x + layer(x)`.

### Why do we need it?
This creates an uninterrupted "highway" bypassing the heavy Matrix Multiplication. During Backpropagation, the gradient can sprint down this highway all the way to the Embeddings, completely solving the **Vanishing Gradient Problem**. It is arguably the most important trick in modern Deep Learning.

## 3. Layer Normalization (The Equalizer) vs RMSNorm

### The Concept
When you multiply matrices thousands of times, some numbers get huge (e.g., `4502`) and some get tiny (e.g., `-0.0001`). The huge numbers will mathematically drown out the tiny numbers completely. 

**Layer Normalization** fixes this by calculating the mean (average) and variance of the vector, and forcing the numbers to have a mean of 0 and a variance of 1. It "equalizes" the sound levels.

### Modern Upgrade: RMSNorm
The original 2017 Transformer used standard LayerNorm. Modern LLMs (like Llama and Mistral) use **RMSNorm**, which skips calculating the mean entirely. It just scales the variance. This makes the math 20% faster with almost no drop in quality!

### Why do we need it?
Without Normalization, the numbers spin out of control and the loss becomes `NaN` (Not a Number) during training.

In [1]:
import torch
import torch.nn as nn

# Let's write a simple RMSNorm (used in Llama 3!)
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # The model can learn to scale the numbers up or down if it wants to!
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # Calculate the Root Mean Square
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        
        # Normalize by dividing by RMS, then multiply by learned weights
        return (x / rms) * self.weight

dummy_input = torch.tensor([[-500.0, 2.0, 1000.0]])
norm = RMSNorm(dim=3)
print(f"Crazy Input: {dummy_input[0]}")
print(f"Tamed Output: {norm(dummy_input)[0].detach()}")

Crazy Input: tensor([-500.,    2., 1000.])
Tamed Output: tensor([-0.7746,  0.0031,  1.5492])


## 4. The Feed-Forward Network (The Thinker)

### The Concept
After words have exchanged context through Attention (e.g., "bank" realizes it's near "river"), that new information needs to be processed. The Feed-Forward Network (FFN) is standard neural network: it expands the dimension by 4x, runs a non-linear activation (like GELU or ReLU), and shrinks it back down.

### Why do we need it?
This is the **Memory Bank** of the LLM. While Attention is responsible for *routing* information around the sentence, the FFN stores factual knowledge (e.g., "Paris is in France"). Because it expands the dimension 4x, it has a massive amount of parameters to memorize training data.

In [2]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, d_model)
        self.activation = nn.GELU() # Often used instead of ReLU nowadays

    def forward(self, x):
        # Expand, Activate, Shrink
        return self.w2(self.activation(self.w1(x)))


## 5. Building the Full Encoder Block (Pre-Norm)

### The Concept
We have everything! Let's combine them.

> **Note**: The 2017 paper did Normalization *after* Attention (`x = Norm(x + Attention(x))`). 
> We will use the modern **Pre-Norm** standard (`x = x + Attention(Norm(x))`), which makes training significantly more stable.

In [3]:
import sys, os
sys.path.append(os.path.abspath("../"))
# We will mock the MultiHeadAttention for this block to run standalone cleanly,
# though in a real script we would import the one from Notebook 05!
class MockMultiHeadAttention(nn.Module):
    def __init__(self, d_model): 
        super().__init__()
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x): 
        return self.proj(x)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, hidden_dim: int):
        super().__init__()
        
        # 1. The Attention Block
        self.attention = MockMultiHeadAttention(d_model)
        self.norm1 = RMSNorm(d_model)
        
        # 2. The Thought Block
        self.ffn = FeedForward(d_model, hidden_dim)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x):
        # --- Step 1: Attention --- 
        # Modern Pre-Norm Architecture with Residual Highway
        x = x + self.attention(self.norm1(x))
        
        # --- Step 2: Feed Forward ---
        # Another Pre-Norm and Residual Highway
        x = x + self.ffn(self.norm2(x))
        
        return x

# --- TEST TIME ---
# Pretend we have a batch of 2 sentences, 5 words each, mapped to 128 dimensions
batch_size, seq_len, d_model = 2, 5, 128
dummy_embeds = torch.randn(batch_size, seq_len, d_model)

encoder_layer = TransformerEncoderBlock(d_model=128, num_heads=8, hidden_dim=512) # Hidden dim is usually 4x d_model

output = encoder_layer(dummy_embeds)

print(f"Input Shape:  {dummy_embeds.shape}")
print(f"Output Shape: {output.shape} (Dimensions stay perfectly intact!)")

Input Shape:  torch.Size([2, 5, 128])
Output Shape: torch.Size([2, 5, 128]) (Dimensions stay perfectly intact!)
